# English Premier League Match Outcome Prediction

This notebook implements an end-to-end machine learning system to predict the outcome of English Premier League (EPL) football matches. Using the historical match and shooting statistics dataset (`matches.csv`), we perform exploratory data analysis, data cleaning, temporal feature engineering, rolling window momentum aggregation, and supervised classification with Random Forests.

Finally, we apply a two-sided prediction reconciliation technique that cross-references predictions from both competing clubs to substantially improve precision.

## 1. Environment Setup and Data Ingestion

In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score

In [2]:
matches = pd.read_csv("matches.csv", index_col=0)
matches.head()

,date,time,comp,round,day,venue,result,gf,ga,opponent,xg,xga,poss,attendance,captain,formation,referee,match report,notes,sh,sot,dist,fk,pk,pkatt,season,team
1,2021-08-15,16:30,Premier League,Matchweek 1,Sun,Away,L,0.0,1.0,Tottenham,1.9,1.3,64.0,58262.0,Fernandinho,4-3-3,Anthony Taylor,Match Report,NaN,18.0,4.0,16.9,1.0,0.0,0.0,2022,Manchester City
2,2021-08-21,15:00,Premier League,Matchweek 2,Sat,Home,W,5.0,0.0,Norwich City,2.7,0.1,67.0,51437.0,İlkay Gündoğan,4-3-3,Graham Scott,Match Report,NaN,16.0,4.0,17.3,1.0,0.0,0.0,2022,Manchester City
3,2021-08-28,12:30,Premier League,Matchweek 3,Sat,Home,W,5.0,0.0,Arsenal,3.8,0.1,80.0,52276.0,İlkay Gündoğan,4-3-3,Martin Atkinson,Match Report,NaN,25.0,10.0,14.3,0.0,0.0,0.0,2022,Manchester City
4,2021-09-11,15:00,Premier League,Matchweek 4,Sat,Away,W,1.0,0.0,Leicester City,2.9,0.8,61.0,32087.0,İlkay Gündoğan,4-3-3,Paul Tierney,Match Report,NaN,25.0,8.0,14.0,0.0,0.0,0.0,2022,Manchester City
6,2021-09-18,15:00,Premier League,Matchweek 5,Sat,Home,D,0.0,0.0,Southampton,1.1,0.4,63.0,52698.0,Fernandinho,4-3-3,Jonathan Moss,Match Report,NaN,16.0,1.0,15.7,1.0,0.0,0.0,2022,Manchester City


In [3]:
matches.shape

(1389, 27)

## 2. Exploratory Data Inspection and Completeness Analysis

In a standard Premier League season, 20 squads play 38 matches each, totaling 760 match entries (380 unique fixtures with 2 perspective rows per match). Across two full seasons, we expect $2 \times 20 \times 38 = 1520$ match records.

In [4]:
# Expected total match perspectives across 2 seasons
2 * 20 * 38

1520

In [5]:
# Inspect match record distribution across clubs
matches["team"].value_counts()

team
Southampton                 72
Brighton and Hove Albion    72
Manchester United           72
West Ham United             72
Newcastle United            72
Burnley                     71
Leeds United                71
Crystal Palace              71
Manchester City             71
Wolverhampton Wanderers     71
Tottenham Hotspur           71
Arsenal                     71
Leicester City              70
Chelsea                     70
Aston Villa                 70
Everton                     70
Liverpool                   38
Fulham                      38
West Bromwich Albion        38
Sheffield United            38
Brentford                   34
Watford                     33
Norwich City                33
Name: count, dtype: int64

In [6]:
# Inspect Liverpool fixtures to diagnose missing 2021-2022 records
matches[matches["team"] == "Liverpool"].sort_values("date")

In [7]:
matches["round"].value_counts()

round
Matchweek 1     39
Matchweek 16    39
Matchweek 34    39
Matchweek 32    39
Matchweek 31    39
Matchweek 29    39
Matchweek 28    39
Matchweek 26    39
Matchweek 25    39
Matchweek 24    39
Matchweek 23    39
Matchweek 2     39
Matchweek 19    39
Matchweek 17    39
Matchweek 20    39
Matchweek 15    39
Matchweek 5     39
Matchweek 3     39
Matchweek 13    39
Matchweek 12    39
Matchweek 4     39
Matchweek 11    39
Matchweek 10    39
Matchweek 9     39
Matchweek 8     39
Matchweek 14    39
Matchweek 7     39
Matchweek 6     39
Matchweek 30    37
Matchweek 27    37
Matchweek 22    37
Matchweek 21    37
Matchweek 18    37
Matchweek 33    32
Matchweek 35    20
Matchweek 36    20
Matchweek 37    20
Matchweek 38    20
Name: count, dtype: int64

In [8]:
matches.dtypes

date             object
time             object
comp             object
round            object
day              object
venue            object
result           object
gf              float64
ga              float64
opponent         object
xg              float64
xga             float64
poss            float64
attendance      float64
captain          object
formation        object
referee          object
match report     object
notes           float64
sh              float64
sot             float64
dist            float64
fk              float64
pk              float64
pkatt           float64
season            int64
team             object
dtype: object

## 3. Data Cleaning and Target Variable Formulation

We remove redundant columns (`comp`, `notes`), convert the match `date` string to `datetime64[ns]`, and formulate the binary classification target variable:
$$\text{target} = \begin{cases} 1 & \text{if Result = 'W'} \\ 0 & \text{otherwise (Draw or Loss)} \end{cases}$$

In [9]:
del matches["comp"]
del matches["notes"]
matches["date"] = pd.to_datetime(matches["date"])
matches["target"] = (matches["result"] == "W").astype("int")
matches

,date,time,round,day,venue,result,gf,ga,opponent,xg,xga,poss,attendance,captain,formation,referee,match report,sh,sot,dist,fk,pk,pkatt,season,team,target
1,2021-08-15,16:30,Matchweek 1,Sun,Away,L,0.0,1.0,Tottenham,1.9,1.3,64.0,58262.0,Fernandinho,4-3-3,Anthony Taylor,Match Report,18.0,4.0,16.9,1.0,0.0,0.0,2022,Manchester City,0
2,2021-08-21,15:00,Matchweek 2,Sat,Home,W,5.0,0.0,Norwich City,2.7,0.1,67.0,51437.0,İlkay Gündoğan,4-3-3,Graham Scott,Match Report,16.0,4.0,17.3,1.0,0.0,0.0,2022,Manchester City,1
3,2021-08-28,12:30,Matchweek 3,Sat,Home,W,5.0,0.0,Arsenal,3.8,0.1,80.0,52276.0,İlkay Gündoğan,4-3-3,Martin Atkinson,Match Report,25.0,10.0,14.3,0.0,0.0,0.0,2022,Manchester City,1
4,2021-09-11,15:00,Matchweek 4,Sat,Away,W,1.0,0.0,Leicester City,2.9,0.8,61.0,32087.0,İlkay Gündoğan,4-3-3,Paul Tierney,Match Report,25.0,8.0,14.0,0.0,0.0,0.0,2022,Manchester City,1
6,2021-09-18,15:00,Matchweek 5,Sat,Home,D,0.0,0.0,Southampton,1.1,0.4,63.0,52698.0,Fernandinho,4-3-3,Jonathan Moss,Match Report,16.0,1.0,15.7,1.0,0.0,0.0,2022,Manchester City,0
8,2021-09-25,12:30,Matchweek 6,Sat,Away,W,1.0,0.0,Chelsea,1.7,0.3,60.0,40036.0,Rúben Dias,4-3-3,Michael Oliver,Match Report,15.0,3.0,16.5,0.0,0.0,0.0,2022,Manchester City,1
10,2021-10-03,16:30,Matchweek 7,Sun,Away,D,2.0,2.0,Liverpool,1.2,1.0,51.0,53102.0,Rúben Dias,4-3-3,Paul Tierney,Match Report,12.0,3.0,16.8,1.0,0.0,0.0,2022,Manchester City,0
11,2021-10-16,15:00,Matchweek 8,Sat,Home,W,2.0,0.0,Burnley,1.9,1.0,69.0,52711.0,Kevin De Bruyne,4-3-3,Martin Atkinson,Match Report,16.0,6.0,14.6,0.0,0.0,0.0,2022,Manchester City,1
13,2021-10-23,17:30,Matchweek 9,Sat,Away,W,4.0,1.0,Brighton,3.9,1.3,52.0,31215.0,İlkay Gündoğan,4-3-3,Kevin Friend,Match Report,23.0,13.0,15.7,1.0,0.0,0.0,2022,Manchester City,1
15,2021-10-30,15:00,Matchweek 10,Sat,Home,L,0.0,2.0,Crystal Palace,1.1,1.1,68.0,53014.0,Rúben Dias,4-3-3,Andre Marriner,Match Report,14.0,3.0,17.0,0.0,0.0,0.0,2022,Manchester City,0


## 4. Categorical Encoding and Temporal Feature Extraction

We extract quantitative predictor features from the fixture metadata:
- `venue_code`: Categorical encoding of venue (Home = 1, Away = 0)
- `opp_code`: Categorical integer encoding of the opposing club
- `hour`: Match kick-off hour extracted from time string
- `day_code`: Integer representation of the day of the week (Monday = 0, Sunday = 6)

In [10]:
matches["venue_code"] = matches["venue"].astype("category").cat.codes
matches["opp_code"] = matches["opponent"].astype("category").cat.codes
matches["hour"] = matches["time"].str.replace(":.+", "", regex=True).astype("int")
matches["day_code"] = matches["date"].dt.dayofweek
matches

,date,time,round,day,venue,result,gf,ga,opponent,xg,xga,poss,attendance,captain,formation,referee,match report,sh,sot,dist,fk,pk,pkatt,season,team,target,venue_code,opp_code,hour,day_code
1,2021-08-15,16:30,Matchweek 1,Sun,Away,L,0.0,1.0,Tottenham,1.9,1.3,64.0,58262.0,Fernandinho,4-3-3,Anthony Taylor,Match Report,18.0,4.0,16.9,1.0,0.0,0.0,2022,Manchester City,0,0,18,16,6
2,2021-08-21,15:00,Matchweek 2,Sat,Home,W,5.0,0.0,Norwich City,2.7,0.1,67.0,51437.0,İlkay Gündoğan,4-3-3,Graham Scott,Match Report,16.0,4.0,17.3,1.0,0.0,0.0,2022,Manchester City,1,1,15,15,5
3,2021-08-28,12:30,Matchweek 3,Sat,Home,W,5.0,0.0,Arsenal,3.8,0.1,80.0,52276.0,İlkay Gündoğan,4-3-3,Martin Atkinson,Match Report,25.0,10.0,14.3,0.0,0.0,0.0,2022,Manchester City,1,1,0,12,5
4,2021-09-11,15:00,Matchweek 4,Sat,Away,W,1.0,0.0,Leicester City,2.9,0.8,61.0,32087.0,İlkay Gündoğan,4-3-3,Paul Tierney,Match Report,25.0,8.0,14.0,0.0,0.0,0.0,2022,Manchester City,1,0,10,15,5
6,2021-09-18,15:00,Matchweek 5,Sat,Home,D,0.0,0.0,Southampton,1.1,0.4,63.0,52698.0,Fernandinho,4-3-3,Jonathan Moss,Match Report,16.0,1.0,15.7,1.0,0.0,0.0,2022,Manchester City,0,1,17,15,5
8,2021-09-25,12:30,Matchweek 6,Sat,Away,W,1.0,0.0,Chelsea,1.7,0.3,60.0,40036.0,Rúben Dias,4-3-3,Michael Oliver,Match Report,15.0,3.0,16.5,0.0,0.0,0.0,2022,Manchester City,1,0,5,12,5
10,2021-10-03,16:30,Matchweek 7,Sun,Away,D,2.0,2.0,Liverpool,1.2,1.0,51.0,53102.0,Rúben Dias,4-3-3,Paul Tierney,Match Report,12.0,3.0,16.8,1.0,0.0,0.0,2022,Manchester City,0,0,11,16,6
11,2021-10-16,15:00,Matchweek 8,Sat,Home,W,2.0,0.0,Burnley,1.9,1.0,69.0,52711.0,Kevin De Bruyne,4-3-3,Martin Atkinson,Match Report,16.0,6.0,14.6,0.0,0.0,0.0,2022,Manchester City,1,1,4,15,5
13,2021-10-23,17:30,Matchweek 9,Sat,Away,W,4.0,1.0,Brighton,3.9,1.3,52.0,31215.0,İlkay Gündoğan,4-3-3,Kevin Friend,Match Report,23.0,13.0,15.7,1.0,0.0,0.0,2022,Manchester City,1,0,3,17,5
15,2021-10-30,15:00,Matchweek 10,Sat,Home,L,0.0,2.0,Crystal Palace,1.1,1.1,68.0,53014.0,Rúben Dias,4-3-3,Andre Marriner,Match Report,14.0,3.0,17.0,0.0,0.0,0.0,2022,Manchester City,0,1,6,15,5


## 5. Baseline Random Forest Model

We partition the dataset temporally into training (fixtures before 2022-01-01) and testing (fixtures in 2022) sets. This temporal split prevents lookahead bias in model evaluation.

In [11]:
rf = RandomForestClassifier(n_estimators=50, min_samples_split=10, random_state=1)
train = matches[matches["date"] < '2022-01-01']
test = matches[matches["date"] > '2022-01-01']
predictors = ["venue_code", "opp_code", "hour", "day_code"]
rf.fit(train[predictors], train["target"])

RandomForestClassifier(min_samples_split=10, n_estimators=50, random_state=1)

In [12]:
preds = rf.predict(test[predictors])
error = accuracy_score(test["target"], preds)
print(f"Baseline Accuracy: {error:.4f}")

Baseline Accuracy: 0.6123


In [13]:
combined = pd.DataFrame(dict(actual=test["target"], predicted=preds))
pd.crosstab(index=combined["actual"], columns=combined["predicted"])

In [14]:
baseline_precision = precision_score(test["target"], preds)
print(f"Baseline Precision: {baseline_precision:.4f}")

Baseline Precision: 0.4746


## 6. Feature Engineering: Rolling Performance Averages

To capture club form and tactical momentum, we compute 3-match rolling averages of key performance metrics strictly closed on the left (`closed='left'`) to prevent lookahead leakage:
- Goals Scored (`gf`) and Goals Conceded (`ga`)
- Total Shots (`sh`) and Shots on Target (`sot`)
- Average Shot Distance (`dist`)
- Free Kicks (`fk`), Penalty Kicks (`pk`), and Penalty Attempts (`pkatt`)

In [15]:
grouped_matches = matches.groupby("team")
group = grouped_matches.get_group("Manchester City").sort_values("date")

def rolling_averages(group, cols, new_cols):
    group = group.sort_values("date")
    rolling_stats = group[cols].rolling(3, closed='left').mean()
    group[new_cols] = rolling_stats
    group = group.dropna(subset=new_cols)
    return group

cols = ["gf", "ga", "sh", "sot", "dist", "fk", "pk", "pkatt"]
new_cols = [f"{c}_rolling" for c in cols]

rolling_averages(group, cols, new_cols)

,date,time,round,day,venue,result,gf,ga,opponent,xg,xga,poss,attendance,captain,formation,referee,match report,sh,sot,dist,fk,pk,pkatt,season,team,target,venue_code,opp_code,hour,day_code,gf_rolling,ga_rolling,sh_rolling,sot_rolling,dist_rolling,fk_rolling,pk_rolling,pkatt_rolling
5,2020-10-17,17:30,Matchweek 5,Sat,Home,W,1.0,0.0,Arsenal,1.5,0.9,59.0,NaN,Raheem Sterling,3-5-2,Chris Kavanagh,Match Report,13.0,5.0,16.8,0.0,0.0,0.0,2021,Manchester City,1,1,0,17,5,2.000000,2.333333,17.333333,4.666667,18.900000,1.333333,0.333333,0.333333
7,2020-10-24,12:30,Matchweek 6,Sat,Away,D,1.0,1.0,West Ham,1.1,0.5,70.0,NaN,Raheem Sterling,4-3-3,Anthony Taylor,Match Report,14.0,7.0,20.3,1.0,0.0,0.0,2021,Manchester City,0,0,21,12,5,1.333333,2.000000,17.333333,3.666667,17.733333,0.666667,0.000000,0.000000
9,2020-10-31,12:30,Matchweek 7,Sat,Away,W,1.0,0.0,Sheffield Utd,1.5,0.5,64.0,NaN,Kevin De Bruyne,4-3-3,Michael Oliver,Match Report,16.0,8.0,18.3,2.0,0.0,0.0,2021,Manchester City,1,0,16,12,5,1.000000,0.666667,16.666667,4.333333,18.233333,0.666667,0.000000,0.000000
11,2020-11-08,16:30,Matchweek 8,Sun,Home,D,1.0,1.0,Liverpool,1.6,1.4,54.0,NaN,Kevin De Bruyne,4-2-3-1,Craig Pawson,Match Report,6.0,2.0,19.5,0.0,0.0,1.0,2021,Manchester City,0,1,11,16,6,1.000000,0.333333,14.333333,6.666667,18.466667,1.000000,0.000000,0.000000
12,2020-11-21,17:30,Matchweek 9,Sat,Away,L,0.0,2.0,Tottenham,1.3,0.7,67.0,NaN,Kevin De Bruyne,4-3-3,Mike Dean,Match Report,21.0,5.0,15.6,0.0,0.0,0.0,2021,Manchester City,0,0,18,17,5,1.000000,0.666667,12.000000,5.666667,19.366667,1.000000,0.000000,0.333333
14,2020-11-28,15:00,Matchweek 10,Sat,Home,W,5.0,0.0,Burnley,2.8,0.4,68.0,NaN,Kevin De Bruyne,4-2-3-1,Lee Mason,Match Report,19.0,7.0,15.8,0.0,0.0,0.0,2021,Manchester City,1,1,4,15,5,0.666667,1.000000,14.333333,5.000000,17.800000,0.666667,0.000000,0.333333
16,2020-12-05,15:00,Matchweek 11,Sat,Home,W,2.0,0.0,Fulham,3.0,0.2,69.0,NaN,Kevin De Bruyne,4-2-3-1,Jonathan Moss,Match Report,15.0,5.0,16.0,0.0,1.0,1.0,2021,Manchester City,1,1,8,15,5,2.000000,1.000000,15.333333,4.666667,16.966667,0.000000,0.000000,0.333333
18,2020-12-12,17:30,Matchweek 12,Sat,Away,D,0.0,0.0,Manchester Utd,1.2,1.0,54.0,NaN,Fernandinho,4-2-3-1,Chris Kavanagh,Match Report,9.0,2.0,17.4,1.0,0.0,0.0,2021,Manchester City,0,0,13,17,5,2.333333,0.666667,18.333333,5.666667,15.800000,0.000000,0.333333,0.333333
19,2020-12-15,20:00,Matchweek 13,Tue,Home,D,1.0,1.0,West Brom,1.9,0.3,77.0,NaN,Kevin De Bruyne,4-2-3-1,Peter Bankes,Match Report,26.0,7.0,16.6,2.0,0.0,0.0,2021,Manchester City,0,1,20,20,1,2.333333,0.000000,14.333333,4.666667,16.400000,0.333333,0.333333,0.333333
20,2020-12-19,15:00,Matchweek 14,Sat,Away,W,1.0,0.0,Southampton,1.1,1.0,52.0,NaN,Kevin De Bruyne,4-2-3-1,Mike Dean,Match Report,11.0,5.0,19.1,0.0,0.0,0.0,2021,Manchester City,1,0,17,15,5,1.000000,0.333333,16.666667,4.666667,16.666667,1.000000,0.333333,0.333333


In [16]:
matches_rolling = matches.groupby("team").apply(lambda x: rolling_averages(x, cols, new_cols))
matches_rolling = matches_rolling.droplevel('team')
matches_rolling.index = range(matches_rolling.shape[0])
matches_rolling

,date,time,round,day,venue,result,gf,ga,opponent,xg,xga,poss,attendance,captain,formation,referee,match report,sh,sot,dist,fk,pk,pkatt,season,team,target,venue_code,opp_code,hour,day_code,gf_rolling,ga_rolling,sh_rolling,sot_rolling,dist_rolling,fk_rolling,pk_rolling,pkatt_rolling
0,2020-10-04,14:00,Matchweek 4,Sun,Home,W,2.0,1.0,Sheffield Utd,0.4,0.2,63.0,NaN,Pierre-Emerick Aubameyang,4-3-3,Lee Mason,Match Report,6.0,5.0,16.7,0.0,0.0,0.0,2021,Arsenal,1,1,16,14,6,2.000000,1.333333,7.666667,3.666667,14.733333,0.666667,0.000000,0.000000
1,2020-10-17,17:30,Matchweek 5,Sat,Away,L,0.0,1.0,Manchester City,0.9,1.5,41.0,NaN,Pierre-Emerick Aubameyang,3-4-3,Chris Kavanagh,Match Report,11.0,3.0,18.2,2.0,0.0,0.0,2021,Arsenal,0,0,12,17,5,1.666667,1.666667,5.333333,3.666667,15.766667,0.000000,0.000000,0.000000
2,2020-10-25,19:15,Matchweek 6,Sun,Home,L,0.0,1.0,Leicester City,0.9,1.2,57.0,NaN,Pierre-Emerick Aubameyang,3-4-3,Craig Pawson,Match Report,12.0,4.0,13.2,1.0,0.0,0.0,2021,Arsenal,0,1,10,19,6,1.000000,1.666667,7.000000,3.666667,16.733333,0.666667,0.000000,0.000000
3,2020-11-01,16:30,Matchweek 7,Sun,Away,W,1.0,0.0,Manchester Utd,1.1,0.3,47.0,NaN,Pierre-Emerick Aubameyang,3-4-3,Mike Dean,Match Report,6.0,1.0,22.7,0.0,1.0,1.0,2021,Arsenal,1,0,13,16,6,0.666667,1.000000,9.666667,4.000000,16.033333,1.000000,0.000000,0.000000
4,2020-11-08,19:15,Matchweek 8,Sun,Home,L,0.0,3.0,Aston Villa,1.5,1.7,58.0,NaN,Pierre-Emerick Aubameyang,3-4-3,Martin Atkinson,Match Report,13.0,2.0,15.6,0.0,0.0,0.0,2021,Arsenal,0,1,1,19,6,0.333333,0.666667,9.666667,2.666667,18.033333,1.000000,0.333333,0.333333
5,2020-11-22,16:30,Matchweek 9,Sun,Away,D,0.0,0.0,Leeds United,0.9,2.5,35.0,NaN,Pierre-Emerick Aubameyang,4-2-3-1,Anthony Taylor,Match Report,9.0,2.0,18.5,2.0,0.0,0.0,2021,Arsenal,0,0,9,16,6,0.333333,1.333333,10.333333,2.333333,17.166667,0.333333,0.333333,0.333333
6,2020-11-29,19:15,Matchweek 10,Sun,Home,L,1.0,2.0,Wolves,1.2,1.4,56.0,NaN,Pierre-Emerick Aubameyang,4-2-3-1,Michael Oliver,Match Report,13.0,2.0,14.8,2.0,0.0,0.0,2021,Arsenal,0,1,22,19,6,0.333333,1.000000,9.333333,1.666667,18.933333,0.666667,0.333333,0.333333
7,2020-12-06,16:30,Matchweek 11,Sun,Away,L,0.0,2.0,Tottenham,0.8,0.5,68.0,2000.0,Pierre-Emerick Aubameyang,4-4-2,Martin Atkinson,Match Report,12.0,2.0,15.1,1.0,0.0,0.0,2021,Arsenal,0,0,18,16,6,0.333333,1.666667,11.666667,2.000000,16.300000,1.333333,0.000000,0.000000
8,2020-12-13,19:15,Matchweek 12,Sun,Home,L,0.0,1.0,Burnley,1.5,0.9,65.0,2000.0,Pierre-Emerick Aubameyang,4-2-3-1,Graham Scott,Match Report,18.0,6.0,15.8,0.0,0.0,0.0,2021,Arsenal,0,1,4,19,6,0.333333,1.333333,11.333333,2.000000,16.133333,1.666667,0.000000,0.000000
9,2020-12-16,18:00,Matchweek 13,Wed,Home,D,1.0,1.0,Southampton,0.7,1.1,36.0,NaN,Pierre-Emerick Aubameyang,3-4-3,Paul Tierney,Match Report,9.0,4.0,21.1,0.0,0.0,0.0,2021,Arsenal,0,1,17,18,2,0.333333,1.666667,14.333333,3.333333,15.233333,1.000000,0.000000,0.000000


## 7. Model Re-Training with Rolling Form Predictors

We re-evaluate the Random Forest model incorporating the newly constructed rolling form features alongside our baseline temporal and positional predictors.

In [17]:
def make_predictions(data, predictors):
    train = data[data["date"] < '2022-01-01']
    test = data[data["date"] > '2022-01-01']
    rf.fit(train[predictors], train["target"])
    preds = rf.predict(test[predictors])
    combined = pd.DataFrame(dict(actual=test["target"], predicted=preds), index=test.index)
    error = precision_score(test["target"], preds)
    return combined, error

combined, error = make_predictions(matches_rolling, predictors + new_cols)
print(f"Rolling Features Precision: {error:.4f}")

Rolling Features Precision: 0.6250


In [18]:
combined = combined.merge(matches_rolling[["date", "team", "opponent", "result"]], left_index=True, right_index=True)
combined.head(10)

,actual,predicted,date,team,opponent,result
55,0,0,2022-01-23,Arsenal,Burnley,D
56,1,0,2022-02-10,Arsenal,Wolves,W
57,1,0,2022-02-19,Arsenal,Brentford,W
58,1,1,2022-02-24,Arsenal,Wolves,W
59,1,1,2022-03-06,Arsenal,Watford,W
60,1,1,2022-03-13,Arsenal,Leicester City,W
61,0,1,2022-03-16,Arsenal,Liverpool,L
62,1,0,2022-03-19,Arsenal,Aston Villa,W
63,0,0,2022-04-04,Arsenal,Crystal Palace,L
64,0,0,2022-04-09,Arsenal,Brighton,L


## 8. Two-Sided Fixture Reconciliation

Because every match appears twice in the dataset (once for team $A$ playing team $B$, and once for team $B$ playing team $A$), we can cross-reference the model's independent predictions. If the model predicts team $A$ will win (`predicted_x = 1`) AND simultaneously predicts team $B$ will not win (`predicted_y = 0`), the mutual consensus substantially elevates predictive confidence.

In [19]:
class MissingDict(dict):
    __missing__ = lambda self, key: key

map_values = {
    "Brighton and Hove Albion": "Brighton",
    "Manchester United": "Manchester Utd",
    "Newcastle United": "Newcastle Utd",
    "Tottenham Hotspur": "Tottenham",
    "West Ham United": "West Ham",
    "Wolverhampton Wanderers": "Wolves"
} 
mapping = MissingDict(**map_values)
combined["new_team"] = combined["team"].map(mapping)

merged = combined.merge(combined, left_on=["date", "new_team"], right_on=["date", "opponent"])
merged

,actual_x,predicted_x,date,team_x,opponent_x,result_x,new_team_x,actual_y,predicted_y,team_y,opponent_y,result_y,new_team_y
0,0,0,2022-01-23,Arsenal,Burnley,D,Arsenal,0,0,Burnley,Arsenal,D,Burnley
1,1,0,2022-02-10,Arsenal,Wolves,W,Arsenal,0,0,Wolverhampton Wanderers,Arsenal,L,Wolves
2,1,0,2022-02-19,Arsenal,Brentford,W,Arsenal,0,0,Brentford,Arsenal,L,Brentford
3,1,1,2022-02-24,Arsenal,Wolves,W,Arsenal,0,0,Wolverhampton Wanderers,Arsenal,L,Wolves
4,1,1,2022-03-06,Arsenal,Watford,W,Arsenal,0,0,Watford,Arsenal,L,Watford
5,1,1,2022-03-13,Arsenal,Leicester City,W,Arsenal,0,1,Leicester City,Arsenal,L,Leicester City
6,1,0,2022-03-19,Arsenal,Aston Villa,W,Arsenal,0,0,Aston Villa,Arsenal,L,Aston Villa
7,0,0,2022-04-04,Arsenal,Crystal Palace,L,Arsenal,1,0,Crystal Palace,Arsenal,W,Crystal Palace
8,0,0,2022-04-09,Arsenal,Brighton,L,Arsenal,1,0,Brighton and Hove Albion,Arsenal,W,Brighton
9,0,0,2022-04-16,Arsenal,Southampton,L,Arsenal,1,0,Southampton,Arsenal,W,Southampton


In [20]:
reconciled_counts = merged[(merged["predicted_x"] == 1) & (merged["predicted_y"] == 0)]["actual_x"].value_counts()
print("Reconciled Prediction Distribution:")
print(reconciled_counts)
reconciled_precision = reconciled_counts[1] / (reconciled_counts[1] + reconciled_counts[0])
print(f"\nReconciled Two-Sided Precision: {reconciled_precision:.4%}")

Reconciled Prediction Distribution:
actual_x
1    27
0    13
Name: count, dtype: int64

Reconciled Two-Sided Precision: 67.5000%


## 9. Conclusion and Findings

Through iterative engineering and evaluation, the predictive performance evolved as follows:

| Stage | Predictors | Metric | Score |
|---|---|---|---|
| **Baseline** | `venue_code`, `opp_code`, `hour`, `day_code` | Accuracy | **61.23%** |
| **Baseline** | `venue_code`, `opp_code`, `hour`, `day_code` | Precision | **47.46%** |
| **Rolling Form** | Baseline + 8 rolling averages ($k=3$) | Precision | **62.50%** |
| **Reconciled Consensus** | Dual-perspective consensus agreement | Precision | **67.50%** |

Reconciling predictions from opposing viewpoints eliminated conflicting signals and yielded a **67.50% precision rate** on predicted victories.